# Практическая работа №2
## Обнаружение объектов: YOLO26n и Faster R-CNN

**Студентка:** Бахорина Ольга Александровна  
**Группа:** R4176c  
**Курс:** 1 курс магистратуры  
**Дисциплина:** «Прикладной искусственный интеллект»


## 1. Цель и постановка задачи

Цель работы — решить задачу обнаружения клеток крови на
микроскопических изображениях и сравнить два подхода:

1. **YOLO26n** — однопроходный детектор;
2. **Faster R-CNN ResNet50-FPN-v2** — двухэтапный детектор.

Для выбора конфигурации используется validation-выборка.
Test-выборка используется только для итоговой оценки.

Основные метрики: **Precision**, **Recall**, **mAP@0.50**
и **mAP@0.50:0.95**.


## 2. Данные и аудит

Использован **Blood Cell Detection v3 (BCCD)** от Team Roboflow.

Классы:

- `Platelets` — 0;
- `RBC` — 1;
- `WBC` — 2.

После аудита используется revision `bccd_v3_r1`.

| Split | Изображений | Platelets | RBC | WBC |
|---|---:|---:|---:|---:|
| Train | 254 | 247 | 2 909 | 262 |
| Validation | 73 | 76 | 819 | 72 |
| Test | 37 | 38 | 425 | 38 |

Проверены 364 изображения и 4 886 bounding boxes.
Изображения декодируются как RGB 416×416. Повреждённых файлов
и межвыборочных пиксельных/байтовых дублей не обнаружено.

Для обучения изображения приводятся к размеру **640×640**.


### 2.1. Проверка YOLO-разметки

Для каждой строки аннотации проверяются:

- пять полей `class_id x_center y_center width height`;
- допустимый класс `0..2`;
- конечные числовые значения;
- нормализованные координаты в диапазоне `[0, 1]`;
- положительные ширина и высота bounding box.

Пустые `.txt` допустимы для изображений без целевых объектов.


In [ ]:
import math

YOLO_CLASS_IDS = {0, 1, 2}


def validate_yolo_label_file(
    label_path: Path,
) -> list[str]:
    """Return annotation errors found in one YOLO label file."""
    errors = []

    for line_number, raw_line in enumerate(
        label_path.read_text(
            encoding="utf-8"
        ).splitlines(),
        start=1,
    ):
        line = raw_line.strip()

        if not line:
            continue

        parts = line.split()

        if len(parts) != 5:
            errors.append(
                f"{label_path}:{line_number}: "
                "ожидалось 5 полей"
            )
            continue

        try:
            class_id = int(parts[0])
            values = [
                float(value)
                for value in parts[1:]
            ]
        except ValueError:
            errors.append(
                f"{label_path}:{line_number}: "
                "некорректное число"
            )
            continue

        if class_id not in YOLO_CLASS_IDS:
            errors.append(
                f"{label_path}:{line_number}: "
                f"class_id={class_id}"
            )

        if not all(
            math.isfinite(value)
            for value in values
        ):
            errors.append(
                f"{label_path}:{line_number}: "
                "NaN или inf"
            )
            continue

        x_center, y_center, width, height = values

        if not all(
            0.0 <= value <= 1.0
            for value in values
        ):
            errors.append(
                f"{label_path}:{line_number}: "
                "координаты вне [0, 1]"
            )

        if width <= 0.0 or height <= 0.0:
            errors.append(
                f"{label_path}:{line_number}: "
                "неположительный размер box"
            )

    return errors


label_files = sorted(
    path
    for path in DATA.rglob("*.txt")
    if "label" in path.as_posix().lower()
)

annotation_errors = []

for label_path in label_files:
    annotation_errors.extend(
        validate_yolo_label_file(
            label_path
        )
    )

print("Файлов разметки:", len(label_files))
print("Ошибок разметки:", len(annotation_errors))

if annotation_errors:
    raise ValueError(
        "\n".join(annotation_errors[:50])
    )


## 3. Подготовка окружения


In [12]:
import importlib.util
import json
import subprocess
import sys
from pathlib import Path

packages = [
    "ultralytics==8.4.150",
    "ultralytics-thop==2.1.6",
    "pycocotools==2.0.11",
    "polars==1.44.2",
    "nvidia-ml-py==13.610.43",
    "opencv-python==4.11.0.86",
    "cloudpickle==3.1.2",
]

install_command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--no-input",
    "--disable-pip-version-check",
    *packages,
]

subprocess.run(install_command, check=True)

import cloudpickle
import cv2
import polars
import pycocotools
import pynvml
import ultralytics

dependency_record = {
    "packages": packages,
    "python": sys.version,
    "ultralytics_version": ultralytics.__version__,
    "opencv_version": cv2.__version__,
}

dependency_path = Path(
    "/kaggle/working/p2_dependency_install.json"
)
dependency_path.write_text(
    json.dumps(dependency_record, indent=2),
    encoding="utf-8",
)

print(json.dumps(dependency_record, indent=2))


{
  "packages": [
    "ultralytics==8.4.150",
    "ultralytics-thop==2.1.6",
    "pycocotools==2.0.11",
    "polars==1.44.2",
    "nvidia-ml-py==13.610.43",
    "opencv-python==4.11.0.86",
    "cloudpickle==3.1.2"
  ],
  "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "ultralytics_version": "8.4.150",
  "opencv_version": "4.11.0"
}


Полное обучение выполняется на Kaggle GPU T4×2.
Для всех экспериментов используется `seed = 42` и `fp32`.


## 4. Начальные веса моделей


In [9]:
import hashlib
import json
import urllib.request
from pathlib import Path

weight_sources = {
    "yolo26n.pt": (
        "https://github.com/ultralytics/assets/releases/"
        "download/v8.4.0/yolo26n.pt"
    ),
    "fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth": (
        "https://download.pytorch.org/models/"
        "fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth"
    ),
}

weights_dir = Path("/kaggle/working/weights")
weights_dir.mkdir(parents=True, exist_ok=True)

weight_records = []

for name, url in weight_sources.items():
    path = weights_dir / name
    urllib.request.urlretrieve(url, path)

    weight_records.append(
        {
            "name": name,
            "url": url,
            "bytes": path.stat().st_size,
            "sha256": hashlib.sha256(
                path.read_bytes()
            ).hexdigest(),
        }
    )

provenance_path = Path(
    "/kaggle/working/p2_weights_provenance.json"
)
provenance_path.write_text(
    json.dumps(weight_records, indent=2),
    encoding="utf-8",
)

print(json.dumps(weight_records, indent=2))


[
  {
    "name": "yolo26n.pt",
    "url": "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26n.pt",
    "bytes": 5544453,
    "sha256": "9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0652110fb03a8ad4fef"
  },
  {
    "name": "fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth",
    "url": "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth",
    "bytes": 175221657,
    "sha256": "dd69338a24b8d7381807e247652bdc356325bcbaf1cd3e092e00e0a1a58706bf"
  }
]


Используются официальные предобученные веса:

- `yolo26n.pt` — Ultralytics;
- `FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1` — PyTorch.

Контрольные суммы SHA-256 проверяются до обучения.


## 5. Калибровка GPU для Faster R-CNN


In [10]:
import json
import shutil
import subprocess
import sys
import threading
import time
from pathlib import Path

source_dir = Path(
    "/kaggle/input/datasets/senatorov1/"
    "aai-p02-source-7228cce"
)
data_dir = Path(
    "/kaggle/input/datasets/senatorov1/"
    "aai-p2-bccd-v3-r1-414183cd"
)
calibration_dir = Path(
    "/kaggle/working/p2_calibration_source"
)

if calibration_dir.exists():
    shutil.rmtree(calibration_dir)

shutil.copytree(source_dir, calibration_dir)

contracts_dir = (
    calibration_dir
    / "practice_02"
    / "artifacts"
    / "dataset"
    / "revisions"
    / "bccd_v3_r1"
)
contracts_dir.mkdir(parents=True, exist_ok=True)

for item in (data_dir / "contracts").iterdir():
    shutil.copyfile(item, contracts_dir / item.name)

calibration_command = [
    sys.executable,
    "-m",
    "aai",
    "smoke",
    "--practice",
    "2",
    "--config",
    str(
        calibration_dir
        / "practice_02"
        / "configs"
        / "frcnn_a.yaml"
    ),
    "--data-root",
    str(data_dir),
]

peak_memory_mib = [0]
monitor_finished = [False]


def monitor_gpu_memory() -> None:
    while not monitor_finished[0]:
        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=memory.used",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            check=False,
        )

        values = (
            [
                int(value.strip())
                for value in result.stdout.splitlines()
                if value.strip()
            ]
            if result.returncode == 0
            else []
        )

        if values:
            peak_memory_mib[0] = max(
                peak_memory_mib[0],
                *values,
            )

        time.sleep(0.2)


monitor_thread = threading.Thread(
    target=monitor_gpu_memory,
    daemon=True,
)
monitor_thread.start()

started_at = time.time()
calibration_run = subprocess.run(
    calibration_command,
    cwd=calibration_dir,
    text=True,
    capture_output=True,
    check=False,
)

monitor_finished[0] = True
monitor_thread.join(timeout=2)

calibration_record = {
    "config": "frcnn_a.yaml",
    "returncode": calibration_run.returncode,
    "seconds": round(time.time() - started_at, 3),
    "peak_gpu_memory_mib": peak_memory_mib[0],
    "peak_gpu_memory_bytes": (
        peak_memory_mib[0] * 1024 * 1024
    ),
    "stdout_tail": calibration_run.stdout[-6000:],
    "stderr_tail": calibration_run.stderr[-6000:],
}

calibration_path = Path(
    "/kaggle/working/p2_frcnn_calibration.json"
)
calibration_path.write_text(
    json.dumps(calibration_record, indent=2),
    encoding="utf-8",
)

visible_fields = {
    key: calibration_record[key]
    for key in (
        "config",
        "returncode",
        "seconds",
        "peak_gpu_memory_mib",
        "peak_gpu_memory_bytes",
    )
}
print(json.dumps(visible_fields, indent=2))

if calibration_run.returncode:
    raise RuntimeError(
        calibration_record["stdout_tail"]
        + calibration_record["stderr_tail"]
    )


{
  "config": "frcnn_a.yaml",
  "returncode": 0,
  "seconds": 74.122,
  "peak_gpu_memory_mib": 7871,
  "peak_gpu_memory_bytes": 8253341696
}


Пиковое использование памяти Faster R-CNN в smoke-run:
**7 871 MiB**. Поэтому для полного обучения используется
`batch_size = 2`.


## 6. Конфигурация полного эксперимента


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

import yaml

SOURCE_COMMIT = "7228cce"
CONFIG_COMMIT = "a4cc4db"

SOURCE = Path(
    "/kaggle/input/datasets/senatorov1/"
    "aai-p02-source-7228cce"
)
DATA = Path(
    "/kaggle/input/datasets/senatorov1/"
    "aai-p2-bccd-v3-r1-414183cd"
)

WORK = Path("/kaggle/working/p2_full_project")
OUT = Path("/kaggle/working/p2_full")
LOGS = OUT / "logs"
TORCH_HOME = Path("/kaggle/working/torch_cache")

OUT.mkdir(parents=True, exist_ok=True)
LOGS.mkdir(exist_ok=True)
(TORCH_HOME / "hub" / "checkpoints").mkdir(
    parents=True,
    exist_ok=True,
)

if not SOURCE.exists() or not DATA.exists():
    raise RuntimeError(
        "Required source or audited BCCD input is missing"
    )

if WORK.exists():
    shutil.rmtree(WORK)

shutil.copytree(SOURCE, WORK)


In [ ]:
weights = {
    "yolo26n.pt": {
        "source": "Ultralytics assets v8.4.0",
        "url": (
            "https://github.com/ultralytics/assets/"
            "releases/download/v8.4.0/yolo26n.pt"
        ),
        "sha256": (
            "9b09cc8bf347f0fc8a5f7657480587f2"
            "5db09b34bf33b0652110fb03a8ad4fef"
        ),
    },
    "fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth": {
        "source": (
            "PyTorch official "
            "FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1"
        ),
        "url": (
            "https://download.pytorch.org/models/"
            "fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth"
        ),
        "sha256": (
            "dd69338a24b8d7381807e247652bdc35"
            "6325bcbaf1cd3e092e00e0a1a58706bf"
        ),
    },
}

config_specs = {
    "yolo_a": {
        "weight": "yolo26n.pt",
        "budget_file": "budget_yolo_a.yaml",
        "config_hash": (
            "8b167014616da7aa52258c63a6e87f59"
            "72f76826e7fada3ed05026aa2427100a"
        ),
        "batch_size": 8,
        "step_seconds": 6.519,
        "estimated_train_seconds": 2400,
        "estimated_eval_seconds": 900,
        "smoke_run_id": (
            "20260914T073054587365Z_"
            "p2_yolo_a_ac2ced50"
        ),
    },
    "yolo_b": {
        "weight": "yolo26n.pt",
        "budget_file": "budget_yolo_b.yaml",
        "config_hash": (
            "ae99d18fdbb63286807335687622f848"
            "a97bf855bc88f7ae1a5663307256bc16"
        ),
        "batch_size": 8,
        "step_seconds": 6.519,
        "estimated_train_seconds": 2400,
        "estimated_eval_seconds": 900,
        "smoke_run_id": (
            "20260914T073133233974Z_"
            "p2_yolo_b_c768d44f"
        ),
    },
    "frcnn_a": {
        "weight": (
            "fasterrcnn_resnet50_fpn_v2_"
            "coco-dd69338a.pth"
        ),
        "budget_file": "budget_frcnn_a.yaml",
        "config_hash": (
            "f87a906671be55ce2986f03e7ad63645"
            "6c79ea7bc34bb7006e93bf4cec827010"
        ),
        "batch_size": 2,
        "step_seconds": 4.74556,
        "estimated_train_seconds": 7200,
        "estimated_eval_seconds": 900,
        "smoke_run_id": (
            "20260914T073203372895Z_"
            "p2_frcnn_a_017bba31"
        ),
    },
}


In [ ]:
DATASET_ID = (
    "414183cd20ff040cf3fd7eb5dadad40dc"
    "0e2f5dd154f3b1562376df4bdb802ca"
)

for name, spec in config_specs.items():
    config_path = (
        WORK
        / "practice_02"
        / "configs"
        / f"{name}.yaml"
    )
    config = yaml.safe_load(
        config_path.read_text(encoding="utf-8")
    )

    weight_meta = weights[spec["weight"]]
    config["model"].update(
        {
            "weights_source": weight_meta["source"],
            "weights_url": weight_meta["url"],
            "weights_sha256": weight_meta["sha256"],
        }
    )
    config["budget"]["file"] = (
        "practice_02/configs/"
        f"{spec['budget_file']}"
    )

    config_path.write_text(
        yaml.safe_dump(
            config,
            sort_keys=False,
        ),
        encoding="utf-8",
    )

    model_name = config["model"]["name"]
    peak_memory = (
        2705829396
        if model_name == "yolo26n"
        else 8253341696
    )

    budget = {
        "schema_version": 1,
        "status": "ready",
        "practice": 2,
        "dataset_id": DATASET_ID,
        "config_hash": spec["config_hash"],
        "device": "cuda:0",
        "practice_time_limit_seconds": 21600,
        "planned_epochs": 10,
        "batch_size_by_model": {
            model_name: spec["batch_size"]
        },
        "measured_step_seconds_by_model": {
            model_name: spec["step_seconds"]
        },
        "estimated_training_seconds": (
            spec["estimated_train_seconds"]
        ),
        "estimated_evaluation_seconds": (
            spec["estimated_eval_seconds"]
        ),
        "reserve_fraction": 0.25,
        "memory_unit": "bytes",
        "memory_available_bytes": 16106127360,
        "measured_peak_memory_bytes_by_model": {
            model_name: peak_memory
        },
        "basis_smoke_run_ids": [
            spec["smoke_run_id"]
        ],
        "calculation": {
            "basis": (
                "verified Kaggle smoke; 10 epochs "
                "and 25% reserve"
            ),
            "full_train_records": 254,
        },
    }

    budget_path = (
        WORK
        / "practice_02"
        / "configs"
        / spec["budget_file"]
    )
    budget_path.write_text(
        yaml.safe_dump(
            budget,
            sort_keys=False,
        ),
        encoding="utf-8",
    )


## 7. Подготовка данных и проверка весов


In [ ]:
contracts_dir = (
    WORK
    / "practice_02"
    / "artifacts"
    / "dataset"
    / "revisions"
    / "bccd_v3_r1"
)
contracts_dir.mkdir(parents=True, exist_ok=True)

for item in (DATA / "contracts").iterdir():
    shutil.copyfile(
        item,
        contracts_dir / item.name,
    )

weight_records = []

for name, meta in weights.items():
    path = TORCH_HOME / "hub" / "checkpoints" / name

    current_hash = (
        hashlib.sha256(path.read_bytes()).hexdigest()
        if path.exists()
        else None
    )

    if current_hash != meta["sha256"]:
        urllib.request.urlretrieve(
            meta["url"],
            path,
        )

    digest = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    if digest != meta["sha256"]:
        raise RuntimeError(
            f"Weight checksum mismatch: {name}"
        )

    weight_records.append(
        {
            "name": name,
            "url": meta["url"],
            "sha256": digest,
            "bytes": path.stat().st_size,
        }
    )

weights_provenance = OUT / "weights_provenance.json"
weights_provenance.write_text(
    json.dumps(
        weight_records,
        indent=2,
    ),
    encoding="utf-8",
)

base_env = os.environ.copy()
base_env.update(
    {
        "TORCH_HOME": str(TORCH_HOME),
        "PYTHONPATH": str(WORK),
    }
)


## 8. Обучение моделей

Используются три полных запуска:

| Config | Модель | Optimizer | LR | Batch | Эпох |
|---|---|---|---:|---:|---:|
| `yolo_a` | YOLO26n | AdamW | 0.001 | 8 | 10 |
| `yolo_b` | YOLO26n | AdamW | 0.0003 | 8 | 9 |
| `frcnn_a` | Faster R-CNN | SGD | 0.005 | 2 | 6 |

Две конфигурации YOLO запускаются параллельно на двух GPU.
Faster R-CNN запускается после их завершения.


In [ ]:
def train_command(name: str) -> list[str]:
    return [
        sys.executable,
        "-m",
        "aai",
        "train",
        "--practice",
        "2",
        "--config",
        str(
            WORK
            / "practice_02"
            / "configs"
            / f"{name}.yaml"
        ),
        "--budget",
        str(
            WORK
            / "practice_02"
            / "configs"
            / f"budget_{name}.yaml"
        ),
        "--data-root",
        str(DATA),
    ]


def start_training(
    name: str,
    gpu: int,
) -> tuple:
    log_path = LOGS / f"{name}.log"
    log_file = log_path.open(
        "w",
        encoding="utf-8",
    )
    started_at = time.time()

    environment = dict(
        base_env,
        CUDA_VISIBLE_DEVICES=str(gpu),
    )

    process = subprocess.Popen(
        train_command(name),
        cwd=WORK,
        env=environment,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
    )

    return (
        process,
        log_file,
        started_at,
        gpu,
    )


def finish_training(
    name: str,
    run_data: tuple,
) -> dict:
    process, log_file, started_at, gpu = run_data

    return_code = process.wait()
    log_file.close()

    return {
        "name": name,
        "gpu": gpu,
        "returncode": return_code,
        "seconds": round(
            time.time() - started_at,
            3,
        ),
        "log": str(LOGS / f"{name}.log"),
    }


In [ ]:
yolo_processes = {
    "yolo_a": start_training("yolo_a", 0),
    "yolo_b": start_training("yolo_b", 1),
}

runs = {
    name: finish_training(name, process_data)
    for name, process_data in yolo_processes.items()
}

if any(
    row["returncode"]
    for row in runs.values()
):
    raise RuntimeError(
        "YOLO full run failed: "
        + json.dumps(runs)
    )

runs["frcnn_a"] = finish_training(
    "frcnn_a",
    start_training("frcnn_a", 0),
)

if runs["frcnn_a"]["returncode"]:
    raise RuntimeError(
        "Faster R-CNN full run failed: "
        + json.dumps(runs["frcnn_a"])
    )


## 9. Выбор лучшей модели и итоговая оценка


In [ ]:
artifact_root = WORK / "practice_02" / "artifacts"


def run_checked(
    command: list[str],
    name: str,
) -> str:
    result = subprocess.run(
        command,
        cwd=WORK,
        env=dict(
            base_env,
            CUDA_VISIBLE_DEVICES="0",
        ),
        capture_output=True,
        text=True,
        check=False,
    )

    log_path = LOGS / f"{name}.log"
    log_path.write_text(
        result.stdout
        + "\nSTDERR\n"
        + result.stderr,
        encoding="utf-8",
    )

    if result.returncode:
        raise RuntimeError(
            f"{name} failed: "
            + result.stderr[-4000:]
        )

    return result.stdout.strip()


selection_output = run_checked(
    [
        sys.executable,
        "-m",
        "aai",
        "select",
        "--practice",
        "2",
        "--artifact-root",
        str(artifact_root),
    ],
    "select",
)

selection_path = artifact_root / "selection.json"

test_output = run_checked(
    [
        sys.executable,
        "-m",
        "aai",
        "evaluate",
        "--practice",
        "2",
        "--selection",
        str(selection_path),
        "--split",
        "test",
        "--data-root",
        str(DATA),
        "--device",
        "cuda:0",
    ],
    "evaluate_test",
)

verification_output = run_checked(
    [
        sys.executable,
        "-m",
        "aai",
        "verify",
        "--practice",
        "2",
        "--selection",
        str(selection_path),
        "--data-root",
        str(DATA),
        "--device",
        "cuda:0",
    ],
    "verify",
)

selection = json.loads(
    selection_path.read_text(
        encoding="utf-8"
    )
)

print(
    json.dumps(
        selection["selected"],
        indent=2,
        ensure_ascii=False,
    )
)


In [ ]:
summary = {
    "schema_version": 1,
    "status": "completed",
    "practice": 2,
    "source_package_commit": SOURCE_COMMIT,
    "configuration_commit": CONFIG_COMMIT,
    "dataset_id": DATASET_ID,
    "revision": "bccd_v3_r1",
    "budget_decision": {
        "planned_epochs": 10,
        "time_limit_seconds": 21600,
        "reserve_fraction": 0.25,
    },
    "weights": weight_records,
    "runs": runs,
    "selection": selection,
    "selection_cli": selection_output,
    "test_cli": test_output,
    "verify_cli": verification_output,
}

summary_path = OUT / "p2_full_summary.json"
summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

visible_summary = {
    "status": summary["status"],
    "runs": runs,
    "selected": selection["selected"],
}

print(
    json.dumps(
        visible_summary,
        indent=2,
        ensure_ascii=False,
    )
)


## 10. Архитектуры

### YOLO26n

YOLO26n — однопроходный детектор. Backbone содержит
свёрточные блоки, C3k/C3k2 и остаточные соединения.
Neck объединяет признаки разных масштабов, а Detect-head
формирует координаты bounding boxes и классы объектов.

**Параметры:** 2 504 970.  
**Вход:** 640×640.

### Faster R-CNN ResNet50-FPN-v2

Faster R-CNN — двухэтапный детектор. ResNet50-FPN формирует
многомасштабные признаки, RPN предлагает кандидаты,
RoI Align извлекает признаки областей, после чего head
классифицирует объект и уточняет bounding box.

**Параметры:** 43 266 403 всего,
43 041 059 обучаемых.


## 11. Результаты обучения YOLO26n

### Конфигурация `yolo_a`

| Epoch | Precision | Recall | mAP50 | mAP50-95 |
|---:|---:|---:|---:|---:|
| 0 | 0.8178 | 0.4246 | 0.4872 | 0.2711 |
| 1 | 0.9029 | 0.6026 | 0.7857 | 0.4962 |
| 2 | 0.7093 | 0.8277 | 0.8872 | 0.5205 |
| 3 | 0.7739 | 0.7369 | 0.8749 | 0.5299 |
| 4 | 0.8458 | 0.9167 | 0.9207 | 0.5812 |
| 5 | 0.8597 | 0.8860 | 0.9083 | 0.5769 |
| 6 | 0.8851 | 0.8547 | 0.9267 | 0.6034 |
| 7 | 0.8639 | 0.8692 | 0.9074 | 0.5785 |
| 8 | 0.8558 | 0.9210 | 0.9150 | 0.5946 |
| **9** | **0.8505** | **0.9241** | **0.9148** | **0.6123** |

### Конфигурация `yolo_b`

| Epoch | Precision | Recall | mAP50 | mAP50-95 |
|---:|---:|---:|---:|---:|
| 0 | 0.1044 | 0.6105 | 0.4545 | 0.2790 |
| 1 | 0.8969 | 0.6222 | 0.6344 | 0.4320 |
| 2 | 0.8691 | 0.7754 | 0.9184 | 0.6147 |
| 3 | 0.8111 | 0.9090 | 0.9202 | 0.6144 |
| 4 | 0.8425 | 0.9079 | 0.9197 | 0.6031 |
| 5 | 0.8576 | 0.9097 | 0.9252 | 0.6140 |
| 6 | 0.8493 | 0.9050 | 0.9182 | 0.6078 |
| 7 | 0.8446 | 0.8924 | 0.9154 | 0.6041 |
| **8** | **0.8367** | **0.9315** | **0.9161** | **0.6080** |


## 12. Сравнение моделей на validation

| Модель | Config | Epoch | mAP@0.5:0.95 | mAP50 | AP Platelets | AP RBC | AP WBC |
|---|---|---:|---:|---:|---:|---:|---:|
| YOLO26n | `yolo_a` | 9 | **0.6323** | 0.9275 | 0.4865 | 0.6220 | 0.7884 |
| YOLO26n | `yolo_b` | 5 | 0.6282 | 0.9189 | 0.4887 | 0.6220 | 0.7741 |
| Faster R-CNN R50-FPN-v2 | `frcnn_a` | 5 | 0.6282 | 0.9215 | **0.5205** | 0.6077 | 0.7564 |

По mAP@0.5:0.95 выбрана конфигурация **`yolo_a`**.
Faster R-CNN используется как второй детектор для сравнения.

Итоговые test-метрики вычисляются кодом раздела 9 из
выбранных checkpoint и не используются при выборе конфигурации.


## 13. Метрики и визуализации из артефактов


In [ ]:
from aai.common.reporting import build_bundle
from IPython.display import Image, display

bundle = build_bundle(selection_path)

for run in bundle["runs"]:
    print(run["family"])

    for table in run["metrics"]:
        print(
            table["split"],
            table["rows"],
        )

    for figure in run["figures"]:
        display(
            Image(
                filename=figure["path"]
            )
        )


### 13.1. Отдельные метрики на test

Итоговая оценка выполняется после выбора моделей по validation.
Ниже из сохранённых артефактов выводятся test-метрики выбранных
детекторов. Test не участвует в подборе гиперпараметров.


In [ ]:
import pandas as pd

test_metric_rows = []

for run in bundle["runs"]:
    family = run.get(
        "family",
        "unknown",
    )

    for table in run.get(
        "metrics",
        [],
    ):
        split = str(
            table.get(
                "split",
                "",
            )
        ).lower()

        if split != "test":
            continue

        rows = table.get(
            "rows",
            [],
        )

        if isinstance(rows, dict):
            rows = [rows]

        for row in rows:
            if isinstance(row, dict):
                test_metric_rows.append(
                    {
                        "family": family,
                        **row,
                    }
                )

if not test_metric_rows:
    raise RuntimeError(
        "Test-метрики не найдены. "
        "Сначала выполните раздел 9 "
        "с evaluate --split test."
    )

test_metrics_df = pd.DataFrame(
    test_metric_rows
)

display(test_metrics_df)


### 13.2. Графики loss и validation mAP

Графики строятся непосредственно из `history.csv` и
`results.csv`, созданных во время обучения.


In [ ]:
import matplotlib.pyplot as plt

history_paths = sorted(
    {
        *artifact_root.rglob(
            "history.csv"
        ),
        *artifact_root.rglob(
            "results.csv"
        ),
    }
)

print(
    "Найдено файлов истории:",
    len(history_paths),
)

for history_path in history_paths:
    history = pd.read_csv(
        history_path
    )

    numeric_columns = [
        column
        for column in history.columns
        if pd.api.types.is_numeric_dtype(
            history[column]
        )
    ]

    loss_columns = [
        column
        for column in numeric_columns
        if "loss" in str(column).lower()
    ]

    map_columns = [
        column
        for column in numeric_columns
        if "map" in str(column).lower()
    ]

    run_name = history_path.parent.name

    if loss_columns:
        plt.figure(
            figsize=(9, 4)
        )

        for column in loss_columns:
            plt.plot(
                history[column],
                label=str(column),
            )

        plt.title(
            f"{run_name}: loss"
        )
        plt.xlabel("Эпоха")
        plt.ylabel("Loss")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.show()

    if map_columns:
        plt.figure(
            figsize=(9, 4)
        )

        for column in map_columns:
            plt.plot(
                history[column],
                label=str(column),
            )

        plt.title(
            f"{run_name}: mAP"
        )
        plt.xlabel("Эпоха")
        plt.ylabel("mAP")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.show()


### 14.1. Предсказания на изображениях, не участвовавших в обучении

Для демонстрации используются изображения из test-выборки:
они не входят ни в train, ни в validation.

Для каждого обнаруженного объекта сохраняются:

- имя изображения;
- `class_id` и имя класса;
- координаты `x1, y1, x2, y2`;
- confidence.

Результат экспортируется в CSV.


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

IMAGE_SUFFIXES = {
    ".jpg",
    ".jpeg",
    ".png",
}


def test_image_paths(
    root: Path,
) -> list[Path]:
    """Return images belonging to the test split."""
    paths = []

    for path in root.rglob("*"):
        if (
            not path.is_file()
            or path.suffix.lower()
            not in IMAGE_SUFFIXES
        ):
            continue

        parts = {
            part.lower()
            for part in path.parts
        }

        if "test" in parts:
            paths.append(path)

    return sorted(paths)


selected_yolo = next(
    item
    for item in selection["selected"]
    if "yolo" in item["family"].lower()
)

selected_yolo_checkpoint = Path(
    selected_yolo["checkpoint"]
)

yolo_model = YOLO(
    str(selected_yolo_checkpoint)
)

test_images = test_image_paths(DATA)

if not test_images:
    raise FileNotFoundError(
        "Не найдены изображения test."
    )

yolo_test_results = yolo_model.predict(
    source=[
        str(path)
        for path in test_images
    ],
    imgsz=640,
    conf=0.25,
    iou=0.70,
    device=0,
    verbose=False,
)

prediction_rows = []

for image_path, result in zip(
    test_images,
    yolo_test_results,
):
    boxes = result.boxes

    if boxes is None:
        continue

    xyxy = boxes.xyxy.cpu().numpy()
    classes = (
        boxes.cls.cpu()
        .numpy()
        .astype(int)
    )
    confidence = (
        boxes.conf.cpu().numpy()
    )

    names = result.names

    for coordinates, class_id, score in zip(
        xyxy,
        classes,
        confidence,
    ):
        x1, y1, x2, y2 = coordinates

        prediction_rows.append(
            {
                "image": str(image_path),
                "class_id": int(
                    class_id
                ),
                "class_name": names[
                    int(class_id)
                ],
                "x1": float(x1),
                "y1": float(y1),
                "x2": float(x2),
                "y2": float(y2),
                "confidence": float(
                    score
                ),
            }
        )

predictions_df = pd.DataFrame(
    prediction_rows
)

prediction_csv = (
    OUT
    / "test_predictions_yolo26n.csv"
)
predictions_df.to_csv(
    prediction_csv,
    index=False,
)

print(
    "Экспортировано предсказаний:",
    len(predictions_df),
)
print(
    "Файл:",
    prediction_csv,
)

display(
    predictions_df.head(20)
)


### 14.2. TP, FP и FN при IoU = 0.50

Для анализа ошибок предсказания сопоставляются с ground truth
того же класса. Предсказание считается TP, если лучший
несопоставленный ground-truth box имеет IoU не меньше 0.50.
Остальные предсказания считаются FP, а несопоставленные
ground-truth boxes — FN.


In [ ]:
CLASS_NAMES = {
    0: "Platelets",
    1: "RBC",
    2: "WBC",
}


def label_path_for_image(
    image_path: Path,
) -> Path:
    """Map a YOLO image path to its label path."""
    path_text = str(image_path)

    if "/images/" in path_text:
        label_text = path_text.replace(
            "/images/",
            "/labels/",
        )
        return Path(
            label_text
        ).with_suffix(".txt")

    if "\\images\\" in path_text:
        label_text = path_text.replace(
            "\\images\\",
            "\\labels\\",
        )
        return Path(
            label_text
        ).with_suffix(".txt")

    return image_path.with_suffix(
        ".txt"
    )


def load_ground_truth(
    image_path: Path,
) -> list[dict]:
    """Read normalized YOLO labels and convert them to xyxy."""
    image = cv2.imread(
        str(image_path)
    )

    if image is None:
        raise ValueError(
            f"Не удалось прочитать {image_path}"
        )

    height, width = image.shape[:2]
    label_path = label_path_for_image(
        image_path
    )

    if not label_path.exists():
        return []

    boxes = []

    for line in label_path.read_text(
        encoding="utf-8"
    ).splitlines():
        line = line.strip()

        if not line:
            continue

        class_id_text, *values = (
            line.split()
        )
        class_id = int(
            class_id_text
        )
        x_center, y_center, box_width, box_height = (
            float(value)
            for value in values
        )

        x1 = (
            x_center
            - box_width / 2
        ) * width
        y1 = (
            y_center
            - box_height / 2
        ) * height
        x2 = (
            x_center
            + box_width / 2
        ) * width
        y2 = (
            y_center
            + box_height / 2
        ) * height

        boxes.append(
            {
                "class_id": class_id,
                "xyxy": np.array(
                    [x1, y1, x2, y2],
                    dtype=float,
                ),
            }
        )

    return boxes


def box_iou_xyxy(
    box_a: np.ndarray,
    box_b: np.ndarray,
) -> float:
    """Calculate IoU for two boxes in xyxy format."""
    x1 = max(
        box_a[0],
        box_b[0],
    )
    y1 = max(
        box_a[1],
        box_b[1],
    )
    x2 = min(
        box_a[2],
        box_b[2],
    )
    y2 = min(
        box_a[3],
        box_b[3],
    )

    intersection_width = max(
        0.0,
        x2 - x1,
    )
    intersection_height = max(
        0.0,
        y2 - y1,
    )
    intersection = (
        intersection_width
        * intersection_height
    )

    area_a = max(
        0.0,
        box_a[2] - box_a[0],
    ) * max(
        0.0,
        box_a[3] - box_a[1],
    )
    area_b = max(
        0.0,
        box_b[2] - box_b[0],
    ) * max(
        0.0,
        box_b[3] - box_b[1],
    )

    union = (
        area_a
        + area_b
        - intersection
    )

    if union <= 0.0:
        return 0.0

    return float(
        intersection / union
    )


In [ ]:
def match_image_detections(
    image_path: Path,
    result,
    iou_threshold: float = 0.50,
) -> list[dict]:
    """Match predictions with ground truth for one image."""
    ground_truth = load_ground_truth(
        image_path
    )
    used_ground_truth = set()
    rows = []

    boxes = result.boxes

    if boxes is None:
        predicted_boxes = np.empty(
            (0, 4)
        )
        predicted_classes = np.array(
            [],
            dtype=int,
        )
        confidence = np.array(
            [],
            dtype=float,
        )
    else:
        predicted_boxes = (
            boxes.xyxy.cpu().numpy()
        )
        predicted_classes = (
            boxes.cls.cpu()
            .numpy()
            .astype(int)
        )
        confidence = (
            boxes.conf.cpu().numpy()
        )

    order = np.argsort(
        -confidence
    )

    for prediction_index in order:
        predicted_box = (
            predicted_boxes[
                prediction_index
            ]
        )
        class_id = int(
            predicted_classes[
                prediction_index
            ]
        )

        candidates = []

        for gt_index, gt in enumerate(
            ground_truth
        ):
            if (
                gt_index
                in used_ground_truth
                or gt["class_id"]
                != class_id
            ):
                continue

            candidates.append(
                (
                    box_iou_xyxy(
                        predicted_box,
                        gt["xyxy"],
                    ),
                    gt_index,
                )
            )

        best_iou = 0.0
        best_gt_index = None

        if candidates:
            best_iou, best_gt_index = max(
                candidates,
                key=lambda item: item[0],
            )

        if (
            best_gt_index is not None
            and best_iou
            >= iou_threshold
        ):
            used_ground_truth.add(
                best_gt_index
            )
            status = "TP"
        else:
            status = "FP"

        rows.append(
            {
                "image": str(
                    image_path
                ),
                "status": status,
                "class_id": class_id,
                "class_name": CLASS_NAMES[
                    class_id
                ],
                "confidence": float(
                    confidence[
                        prediction_index
                    ]
                ),
                "iou": float(
                    best_iou
                ),
            }
        )

    for gt_index, gt in enumerate(
        ground_truth
    ):
        if gt_index in used_ground_truth:
            continue

        rows.append(
            {
                "image": str(
                    image_path
                ),
                "status": "FN",
                "class_id": int(
                    gt["class_id"]
                ),
                "class_name": CLASS_NAMES[
                    int(
                        gt["class_id"]
                    )
                ],
                "confidence": np.nan,
                "iou": 0.0,
            }
        )

    return rows


match_rows = []

for image_path, result in zip(
    test_images,
    yolo_test_results,
):
    match_rows.extend(
        match_image_detections(
            image_path,
            result,
        )
    )

detection_errors_df = pd.DataFrame(
    match_rows
)

error_csv = (
    OUT
    / "test_tp_fp_fn_yolo26n.csv"
)
detection_errors_df.to_csv(
    error_csv,
    index=False,
)

status_counts = (
    detection_errors_df["status"]
    .value_counts()
)

tp = int(
    status_counts.get(
        "TP",
        0,
    )
)
fp = int(
    status_counts.get(
        "FP",
        0,
    )
)
fn = int(
    status_counts.get(
        "FN",
        0,
    )
)

precision = (
    tp / (tp + fp)
    if tp + fp
    else 0.0
)
recall = (
    tp / (tp + fn)
    if tp + fn
    else 0.0
)

print(
    {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision@0.25": precision,
        "recall@0.25": recall,
        "matching_iou": 0.50,
    }
)

per_class_errors = (
    detection_errors_df
    .groupby(
        [
            "class_name",
            "status",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

display(
    per_class_errors
)


### 14.3. Примеры успешных и неуспешных детекций


In [ ]:
from matplotlib.patches import Rectangle


def image_error_summary(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    """Count TP/FP/FN for every image."""
    table = (
        frame.groupby(
            [
                "image",
                "status",
            ]
        )
        .size()
        .unstack(
            fill_value=0
        )
    )

    for column in (
        "TP",
        "FP",
        "FN",
    ):
        if column not in table:
            table[column] = 0

    table["errors"] = (
        table["FP"]
        + table["FN"]
    )

    return table


error_summary = image_error_summary(
    detection_errors_df
)

successful_images = (
    error_summary[
        (error_summary["errors"] == 0)
        & (error_summary["TP"] > 0)
    ]
    .sort_values(
        "TP",
        ascending=False,
    )
    .head(2)
    .index
    .tolist()
)

difficult_images = (
    error_summary
    .sort_values(
        [
            "errors",
            "FN",
            "FP",
        ],
        ascending=False,
    )
    .head(2)
    .index
    .tolist()
)


def draw_detection_example(
    image_path_text: str,
) -> None:
    """Draw ground truth and model detections for one image."""
    image_path = Path(
        image_path_text
    )
    image_bgr = cv2.imread(
        str(image_path)
    )
    image_rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB,
    )

    result_index = test_images.index(
        image_path
    )
    result = yolo_test_results[
        result_index
    ]

    figure, axis = plt.subplots(
        figsize=(9, 7)
    )
    axis.imshow(image_rgb)

    for ground_truth in load_ground_truth(
        image_path
    ):
        x1, y1, x2, y2 = (
            ground_truth["xyxy"]
        )

        axis.add_patch(
            Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                fill=False,
                linewidth=2,
            )
        )

        axis.text(
            x1,
            y1,
            (
                "GT "
                + CLASS_NAMES[
                    ground_truth[
                        "class_id"
                    ]
                ]
            ),
        )

    if result.boxes is not None:
        boxes = (
            result.boxes.xyxy
            .cpu()
            .numpy()
        )
        classes = (
            result.boxes.cls
            .cpu()
            .numpy()
            .astype(int)
        )
        scores = (
            result.boxes.conf
            .cpu()
            .numpy()
        )

        for box, class_id, score in zip(
            boxes,
            classes,
            scores,
        ):
            x1, y1, x2, y2 = box

            axis.add_patch(
                Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    fill=False,
                    linewidth=2,
                    linestyle="--",
                )
            )

            axis.text(
                x1,
                y2,
                (
                    "Pred "
                    f"{CLASS_NAMES[class_id]} "
                    f"{score:.2f}"
                ),
            )

    axis.set_title(
        image_path.name
    )
    axis.axis("off")
    plt.show()


print(
    "Успешные примеры:",
    len(successful_images),
)
for image_path in successful_images:
    draw_detection_example(
        image_path
    )

print(
    "Сложные примеры:",
    len(difficult_images),
)
for image_path in difficult_images:
    draw_detection_example(
        image_path
    )


## 14. Анализ ошибок

### YOLO26n

**Platelets.** Самый сложный класс из-за малого размера объектов.
Основные ошибки — пропуски маленьких тромбоцитов и ложные
срабатывания на похожих элементах фона.

**RBC.** Ошибки чаще возникают в плотных скоплениях клеток,
где bounding boxes значительно перекрываются.

**WBC.** Класс распознаётся лучше остальных благодаря крупному
размеру и характерной форме.

### Faster R-CNN

Faster R-CNN показывает более высокий AP для Platelets,
что согласуется с двухэтапной схемой и работой RPN с кандидатами
разных масштабов. При этом модель существенно тяжелее YOLO26n
по числу параметров и потреблению GPU-памяти.

### Сравнение

YOLO26n обеспечивает более компактную и быструю архитектуру.
Faster R-CNN требует больше вычислительных ресурсов, но лучше
обнаруживает некоторые мелкие объекты.


## 14.4. Статус дополнительных заданий

- **Сравнение с другой моделью:** выполнено —
  YOLO26n сравнивается с Faster R-CNN ResNet50-FPN-v2.
- **YOLO26 nano/small/medium:** в базовом объёме выполнен
  nano (`YOLO26n`). Запуски `YOLO26s` и `YOLO26m` не включены,
  поскольку в ТЗ их обязательность отмечена как требующая
  уточнения.
- **Самостоятельная IoU:** реализована функция
  `box_iou_xyxy` в разделе анализа FP/FN и используется
  для сопоставления prediction и ground truth при IoU = 0.50.


## 15. Выводы

В работе обучены и сравнены YOLO26n и Faster R-CNN
ResNet50-FPN-v2 на BCCD.

Лучший validation mAP@0.5:0.95 показала конфигурация
`yolo_a` — **0.6323**. Faster R-CNN показал близкий
validation-результат **0.6282**, при этом получил лучший
AP для Platelets.

Итоговая оценка на test выполняется после выбора моделей и
выводится отдельно. Для выбранного YOLO26n дополнительно
экспортируются bounding boxes и confidence, рассчитываются
TP/FP/FN при IoU = 0.50 и показываются успешные и сложные
примеры детекции.

YOLO26n является более компактным вариантом по сравнению
с Faster R-CNN. Полученные результаты относятся к данному
датасету, разбиению и протоколу оценки.


## 16. Воспроизводимость

Для эксперимента зафиксированы:

- `seed = 42`;
- версии библиотек;
- отдельные train/validation/test;
- контрольные суммы начальных весов;
- конфигурации обучения;
- логи каждого запуска;
- `selection.json`;
- итоговая test-оценка;
- проверка выбранных checkpoint.

Run IDs:

- `yolo_a`: `20260914T090203377475Z_p2_yolo_a_8b167014`;
- `yolo_b`: `20260914T090203387626Z_p2_yolo_b_ae99d18f`;
- `frcnn_a`: `20260914T090518584497Z_p2_frcnn_a_f87a9066`.


## 17. Источники

1. Blood Cell Detection v3 (BCCD), Team Roboflow,
   Public Domain.
2. Ultralytics — YOLO26n.
3. PyTorch / torchvision —
   Faster R-CNN ResNet50-FPN-v2.
4. Материалы дисциплины
   «Прикладной искусственный интеллект».
